# CerberusVision Phase 4.1 Kontrollü Devam Eğitimi

Bu notebook Phase 4 temiz QLoRA adaptörünü koruyarak düşük öğrenme oranlı continual fine-tuning uygular. Temel modelden yeni adaptör başlatmaz. Phase 4 adaptörü trainable olarak yüklenir, 269 temiz replay kaydı ile 135 hedefli zor örnek birlikte eğitilir. Benchmark fixture'ları eğitim paketinde bulunmaz.

## 1. Sabitlenmiş ortam

Colab çalışma zamanı türünü A100 GPU olarak seçin. Paket sürümleri Phase 4 koşusuyla aynıdır; böylece parent adapter ile PEFT uyumluluğu korunur.

In [ ]:
%pip install -q --upgrade transformers==5.14.1 trl==1.8.0 peft==0.19.1 accelerate==1.14.0 datasets==5.0.0 bitsandbytes==0.49.2 safetensors

## 2. Drive, A100 ve deney sabitleri

`PACKAGE_DIR` altında veri dosyalarıyla birlikte `phase4_adapter` dizini bulunmalıdır. `RESUME_FROM_MATCHING_CHECKPOINT` yalnız aynı Phase 4.1 koşusu Colab kesintisi nedeniyle yarıda kaldığında etkinleştirilmelidir.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import random
import re
import shutil
import statistics
import torch
import unicodedata

drive.mount('/content/drive')

SEED = 3407
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
BASE_MODEL_REVISION = 'a09a35458c702b33eeacc393d103063234e8bc28'
MAX_SEQUENCE_LENGTH = 2048
PACKAGE_DIR = Path('/content/drive/MyDrive/CerberusVision_Phase4_1_Colab')
PARENT_ADAPTER_DIR = PACKAGE_DIR / 'phase4_adapter'
RESUME_FROM_MATCHING_CHECKPOINT = False

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU bulunamadi')

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
if 'A100' not in GPU_NAME.upper():
    raise RuntimeError(f'Bu sozlesme A100 icin hazirlandi, bulunan GPU: {GPU_NAME}')
if GPU_MEMORY_GB < 35:
    raise RuntimeError(f'Yetersiz GPU bellegi: {GPU_MEMORY_GB:.1f} GB')

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(json.dumps({'gpu': GPU_NAME, 'memory_gb': round(GPU_MEMORY_GB, 2)}, ensure_ascii=False, indent=2))

## 3. Veri, parent adapter ve sızıntı doğrulaması

Manifestteki bütün dosya hash'leri, Phase 4 adapter ağırlığı ve adapter yapılandırması doğrulanır. Train ve validation normalize hash kümeleri ayrık olmalıdır. Hard validation yalnız ana validation kümesinin bir alt kümesi olabilir.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def normalized_input_hash(value):
    normalized = unicodedata.normalize('NFKC', value).casefold()
    normalized = re.sub(r'[^\w]+', ' ', normalized, flags=re.UNICODE)
    normalized = ' '.join(normalized.split())
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

def canonical_record_hash(record):
    serialized = json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    return hashlib.sha256(serialized.encode('utf-8')).hexdigest()

required_files = [
    'continuation_train.jsonl',
    'validation.jsonl',
    'hard_validation.jsonl',
    'selection_manifest.json',
    'phase4_1_contract.json',
    'phase4_1_manifest.json',
]
missing_files = [name for name in required_files if not (PACKAGE_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f'Eksik Phase 4.1 dosyalari: {missing_files}')

manifest = json.loads((PACKAGE_DIR / 'phase4_1_manifest.json').read_text(encoding='utf-8'))
contract = json.loads((PACKAGE_DIR / 'phase4_1_contract.json').read_text(encoding='utf-8'))
for file_name, expected_hash in manifest['files'].items():
    actual_hash = sha256_file(PACKAGE_DIR / file_name)
    if actual_hash != expected_hash:
        raise RuntimeError(f'Hash uyusmazligi {file_name}: {actual_hash}')

parent_model_path = PARENT_ADAPTER_DIR / 'adapter_model.safetensors'
parent_config_path = PARENT_ADAPTER_DIR / 'adapter_config.json'
if sha256_file(parent_model_path) != contract['parent_adapter']['adapter_model_sha256']:
    raise RuntimeError('Phase 4 parent adapter hash uyusmazligi')
if sha256_file(parent_config_path) != contract['parent_adapter']['adapter_config_sha256']:
    raise RuntimeError('Phase 4 parent adapter config hash uyusmazligi')

train_rows = load_jsonl(PACKAGE_DIR / contract['dataset']['continuation_train_file'])
validation_rows = load_jsonl(PACKAGE_DIR / contract['dataset']['validation_file'])
hard_validation_rows = load_jsonl(PACKAGE_DIR / contract['dataset']['hard_validation_file'])
for split_name, rows in [('train', train_rows), ('validation', validation_rows), ('hard_validation', hard_validation_rows)]:
    for row_index, row in enumerate(rows):
        if not {'instructions', 'input', 'output'}.issubset(row):
            raise RuntimeError(f'{split_name} eksik alan, kayit {row_index}')
        parsed_output = json.loads(row['output'])
        if not isinstance(parsed_output, dict):
            raise RuntimeError(f'{split_name} JSON object degil, kayit {row_index}')

train_hashes = {normalized_input_hash(row['input']) for row in train_rows}
validation_hashes = {normalized_input_hash(row['input']) for row in validation_rows}
overlap = train_hashes & validation_hashes
if overlap:
    raise RuntimeError(f'Train validation sizintisi: {len(overlap)}')

validation_record_hashes = {canonical_record_hash(row) for row in validation_rows}
hard_validation_record_hashes = {canonical_record_hash(row) for row in hard_validation_rows}
if not hard_validation_record_hashes.issubset(validation_record_hashes):
    raise RuntimeError('Hard validation ana validation alt kumesi degil')

selection = json.loads((PACKAGE_DIR / 'selection_manifest.json').read_text(encoding='utf-8'))
replay_ratio = selection['source_train_records'] / selection['continuation_train_records']
if not 0.6 <= replay_ratio <= 0.7:
    raise RuntimeError(f'Guvenli olmayan replay orani: {replay_ratio}')

print(json.dumps({'train': len(train_rows), 'validation': len(validation_rows), 'hard_validation': len(hard_validation_rows), 'replay_ratio': replay_ratio, 'overlap': len(overlap), 'parent_adapter_sha256': sha256_file(parent_model_path)}, ensure_ascii=False, indent=2))

## 4. Tokenizer, prompt-completion biçimi ve uzunluk kapısı

Loss yalnız assistant completion tokenlerinde hesaplanır. Tek bir örnek bile 2048 token sınırını aşarsa sessiz truncation yerine eğitim durur.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

SYSTEM_PROMPT = 'You are a shipping instruction document parser. Return only valid JSON matching the requested structure. Use only evidence present in the OCR text. Set absent fields to null and never fabricate values.'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def training_row(row):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': row['instructions'] + '\n\n' + row['input']},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {'prompt': prompt, 'completion': row['output'] + tokenizer.eos_token}

train_formatted = [training_row(row) for row in train_rows]
validation_formatted = [training_row(row) for row in validation_rows]
hard_validation_formatted = [training_row(row) for row in hard_validation_rows]
all_formatted = train_formatted + validation_formatted
token_lengths = [len(tokenizer(row['prompt'] + row['completion'], add_special_tokens=False)['input_ids']) for row in all_formatted]
oversized_indices = [index for index, length in enumerate(token_lengths) if length > MAX_SEQUENCE_LENGTH]
if oversized_indices:
    raise RuntimeError(f'Truncation yasagi ihlali: {oversized_indices[:10]}')

train_dataset = Dataset.from_list(train_formatted)
validation_dataset = Dataset.from_list(validation_formatted)
hard_validation_dataset = Dataset.from_list(hard_validation_formatted)
length_report = {'minimum': min(token_lengths), 'maximum': max(token_lengths), 'mean': statistics.mean(token_lengths), 'p95': sorted(token_lengths)[int(len(token_lengths) * 0.95) - 1], 'oversized': len(oversized_indices)}
print(json.dumps(length_report, ensure_ascii=False, indent=2))

## 5. İzole koşu sözleşmesi ve checkpoint alanı

Veri, parent adapter ve hiperparametreler tek runtime hash altında birleştirilir. Başka bir deneyin checkpoint'i bu koşuya bağlanamaz.

In [ ]:
package_versions = {name: importlib.metadata.version(name) for name in ['transformers', 'trl', 'peft', 'accelerate', 'datasets', 'bitsandbytes']}
runtime_contract = {
    'contract': contract,
    'manifest_sha256': sha256_file(PACKAGE_DIR / 'phase4_1_manifest.json'),
    'parent_adapter_sha256': sha256_file(parent_model_path),
    'gpu': GPU_NAME,
    'gpu_memory_gb': GPU_MEMORY_GB,
    'torch': torch.__version__,
    'packages': package_versions,
    'token_lengths': length_report,
}
runtime_contract_json = json.dumps(runtime_contract, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
runtime_contract_hash = hashlib.sha256(runtime_contract_json.encode('utf-8')).hexdigest()
RUN_ROOT = Path('/content/drive/MyDrive/CerberusVision_Phase4_1_Runs') / runtime_contract_hash[:16]
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
ADAPTER_DIR = RUN_ROOT / 'adapter_best'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
(RUN_ROOT / 'runtime_contract.json').write_text(json.dumps(runtime_contract, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({'runtime_contract_hash': runtime_contract_hash, 'run_root': str(RUN_ROOT)}, ensure_ascii=False, indent=2))

## 6. Phase 4 adapter'ını trainable yükle

Temel Qwen modeli NF4 4-bit yüklenir. K-bit hazırlığından sonra Phase 4 adapter'ı `is_trainable=True` ile bağlanır. Yeni LoRA katmanı oluşturulmaz.

In [ ]:
from peft import PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=BASE_MODEL_REVISION,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    device_map={'': 0},
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
model = PeftModel.from_pretrained(base_model, str(PARENT_ADAPTER_DIR), is_trainable=True)
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
all_parameters = sum(parameter.numel() for parameter in model.parameters())
if trainable_parameters <= 0:
    raise RuntimeError('Trainable Phase 4 adapter parametresi bulunamadi')
model.print_trainable_parameters()
print(json.dumps({'trainable_parameters': trainable_parameters, 'all_parameters': all_parameters, 'trainable_ratio': trainable_parameters / all_parameters}, ensure_ascii=False, indent=2))

## 7. Düşük öğrenme oranlı trainer ve eğitim öncesi baseline

Öğrenme oranı `2e-5`, azami epoch sayısı 2 ve effective batch size 16'dır. Validation her 5 optimizer adımında ölçülür. İki değerlendirmede en az `0.0002` iyileşme yoksa eğitim durur.

In [ ]:
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

training_config = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_steps=3,
    weight_decay=0.01,
    max_grad_norm=0.3,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    bf16=True,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    eval_strategy='steps',
    eval_steps=5,
    save_strategy='steps',
    save_steps=5,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_strategy='steps',
    logging_steps=1,
    logging_first_step=True,
    seed=SEED,
    data_seed=SEED,
    report_to='none',
    max_length=MAX_SEQUENCE_LENGTH,
    packing=False,
    completion_only_loss=True,
    dataset_num_proc=2,
)
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0002)],
)
baseline_metrics = trainer.evaluate(metric_key_prefix='baseline')
baseline_hard_metrics = trainer.evaluate(eval_dataset=hard_validation_dataset, metric_key_prefix='baseline_hard')
print(json.dumps({'validation': baseline_metrics, 'hard_validation': baseline_hard_metrics}, ensure_ascii=False, indent=2))

## 8. Eğitimi başlat ve iki validation görünümünü ölç

Resume yalnız runtime hash ile izole edilmiş bu koşunun checkpoint dizininden yapılır. Eğitim sonunda en iyi checkpoint geri yüklenir ve hem normal hem zor validation loss kaydedilir.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR)) if CHECKPOINT_DIR.exists() else None
if RESUME_FROM_MATCHING_CHECKPOINT and last_checkpoint is None:
    raise RuntimeError('Resume istendi ancak uyumlu Phase 4.1 checkpoint bulunamadi')
resume_checkpoint = last_checkpoint if RESUME_FROM_MATCHING_CHECKPOINT else None
training_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
final_metrics = trainer.evaluate(metric_key_prefix='final')
final_hard_metrics = trainer.evaluate(eval_dataset=hard_validation_dataset, metric_key_prefix='final_hard')
print(json.dumps({'train': training_result.metrics, 'validation': final_metrics, 'hard_validation': final_hard_metrics, 'best_checkpoint': trainer.state.best_model_checkpoint, 'best_metric': trainer.state.best_metric}, ensure_ascii=False, indent=2))

## 9. En iyi adapter ve provenance raporu

Phase 4.1 çıktısı parent adapter dizinine yazılmaz. En düşük validation loss ile geri yüklenen adapter ayrı dizinde kaydedilir.

In [ ]:
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
saved_adapter_path = ADAPTER_DIR / 'adapter_model.safetensors'
if not saved_adapter_path.is_file():
    raise RuntimeError('Phase 4.1 adapter_model.safetensors kaydedilmedi')

training_report = {
    'phase': '4.1',
    'runtime_contract_hash': runtime_contract_hash,
    'base_model': BASE_MODEL,
    'base_model_revision': BASE_MODEL_REVISION,
    'parent_adapter_sha256': sha256_file(parent_model_path),
    'parent_adapter_used': True,
    'fresh_adapter_created': False,
    'resume_checkpoint': resume_checkpoint,
    'best_checkpoint': trainer.state.best_model_checkpoint,
    'best_metric': trainer.state.best_metric,
    'final_global_step': trainer.state.global_step,
    'final_epoch': trainer.state.epoch,
    'baseline_metrics': baseline_metrics,
    'baseline_hard_metrics': baseline_hard_metrics,
    'training_metrics': training_result.metrics,
    'final_metrics': final_metrics,
    'final_hard_metrics': final_hard_metrics,
    'log_history': trainer.state.log_history,
    'output_adapter_sha256': sha256_file(saved_adapter_path),
    'dataset_manifest_sha256': sha256_file(PACKAGE_DIR / 'phase4_1_manifest.json'),
    'acceptance_gate': contract['acceptance_gate'],
}
(RUN_ROOT / 'training_report.json').write_text(json.dumps(training_report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({'adapter': str(ADAPTER_DIR), 'sha256': training_report['output_adapter_sha256']}, ensure_ascii=False, indent=2))

## 10. Üretim sağlık kontrolü

Bu bölüm holdout veya benchmark değildir. Normal validation ve hard validation içinden beşer örnekte greedy üretim, JSON parse ve aşırı tekrar kontrolü yapar.

In [ ]:
def excessive_repetition(value):
    tokens = re.findall(r'\w+|[^\w\s]', value.casefold())
    counts = {}
    for start_index in range(0, max(0, len(tokens) - 23), 8):
        window = tuple(tokens[start_index:start_index + 24])
        counts[window] = counts.get(window, 0) + 1
        if counts[window] >= 4:
            return True
    return False

model.eval()
model.config.use_cache = True
validation_predictions = []
health_rows = [('validation', row) for row in validation_rows[:5]] + [('hard_validation', row) for row in hard_validation_rows[:5]]
for split_name, row in health_rows:
    formatted = training_row(row)
    inputs = tokenizer(formatted['prompt'], return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=1024, do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    completion_tokens = generated[0, inputs['input_ids'].shape[1]:]
    completion = tokenizer.decode(completion_tokens, skip_special_tokens=True)
    try:
        parsed = json.loads(completion)
        json_valid = isinstance(parsed, dict)
        parse_error = None
    except Exception as error:
        json_valid = False
        parse_error = f'{type(error).__name__}: {error}'
    validation_predictions.append({'split': split_name, 'input_sha256': normalized_input_hash(row['input']), 'json_valid': json_valid, 'excessive_repetition': excessive_repetition(completion), 'parse_error': parse_error, 'completion': completion})

invalid_predictions = [item for item in validation_predictions if not item['json_valid'] or item['excessive_repetition']]
(RUN_ROOT / 'validation_predictions.json').write_text(json.dumps(validation_predictions, ensure_ascii=False, indent=2), encoding='utf-8')
if invalid_predictions:
    raise RuntimeError(f'Uretim saglik kontrolu basarisiz: {len(invalid_predictions)}')
print(json.dumps({'checked': len(validation_predictions), 'invalid': len(invalid_predictions)}, ensure_ascii=False, indent=2))

## 11. Teslim arşivi

En iyi adapter, tokenizer, runtime sözleşmesi, eğitim raporu ve sağlık çıktıları tek ZIP içine alınır. Checkpoint'ler arşive dahil edilmez.

In [ ]:
DELIVERY_DIR = RUN_ROOT / 'delivery'
if DELIVERY_DIR.exists():
    shutil.rmtree(DELIVERY_DIR)
DELIVERY_DIR.mkdir(parents=True)
shutil.copytree(ADAPTER_DIR, DELIVERY_DIR / 'adapter_best')
for artifact_name in ['runtime_contract.json', 'training_report.json', 'validation_predictions.json']:
    shutil.copy2(RUN_ROOT / artifact_name, DELIVERY_DIR / artifact_name)
shutil.copy2(PACKAGE_DIR / 'phase4_1_contract.json', DELIVERY_DIR / 'phase4_1_contract.json')
shutil.copy2(PACKAGE_DIR / 'phase4_1_manifest.json', DELIVERY_DIR / 'phase4_1_manifest.json')
archive_path = shutil.make_archive(str(RUN_ROOT / f"{contract['run_name']}_delivery"), 'zip', DELIVERY_DIR)
print(json.dumps({'archive': archive_path, 'sha256': sha256_file(Path(archive_path))}, ensure_ascii=False, indent=2))

## 12. Eğitim sonrası karar kapısı

Validation loss tek başına üretim onayı değildir. Teslim ZIP'i projeye entegre edildikten sonra donmuş 13-vaka benchmark iki deterministik tekrarla çalıştırılmalıdır. Kabul için precision en az yüzde 58, recall en az yüzde 90, F1 en az yüzde 69, XSD 13/13, inference error 0 ve deterministik mismatch 0 olmalıdır. Bu koşullardan biri sağlanmazsa Phase 4 adapter üretim seçeneği olarak korunur ve Phase 4.1 varsayılan yapılmaz.